# EmbeddingGemma 2 with MLX-VLM

Create **768-dimensional embeddings** for text, images, audio, video, and composed
inputs on Apple Silicon. This notebook covers retrieval, smaller Matryoshka
vectors, batched encoding, and multimodal inputs using the local MLX-VLM processor.

Embeddings are L2-normalized: their dot product is cosine similarity.
A larger score means a closer match, **not a probability**.

## Setup

Use a Python notebook kernel in the environment where this checkout is installed.
From the repository root, install it with `python -m pip install -e .`.
The processor needs no Torch, TorchVision, TorchAudio, TorchCodec, or PyAV.

Run the cells in order from the repository root or the `examples` directory.
Images and video come from this repository; the audio cell downloads a small,
pinned public speech clip to the Hugging Face cache.

This notebook uses `nativ-community/embeddinggemma-2`.
The first model load downloads weights when they are not already cached.

In [ ]:
from pathlib import Path

import mlx.core as mx
import numpy as np
from huggingface_hub import hf_hub_download
from PIL import Image

from mlx_vlm import load

MODEL_ID = "nativ-community/embeddinggemma-2"
model, processor = load(MODEL_ID)

EXAMPLES_DIR = Path.cwd()
if not (EXAMPLES_DIR / "images" / "cats.jpg").is_file():
    EXAMPLES_DIR = EXAMPLES_DIR / "examples"
if not (EXAMPLES_DIR / "images" / "cats.jpg").is_file():
    raise FileNotFoundError(
        "Run this notebook from the repository root or examples directory."
    )

print("Model:", MODEL_ID)
print("Processor:", type(processor).__name__)

## 1. Text retrieval

Use `load` to obtain both the model and processor, then tokenize text with
`processor.tokenizer`. Retrieval queries and documents use different task prefixes.
`text_embeds` contains one vector per input; `last_hidden_state` contains token vectors.

In [ ]:
query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size.",
    "Mars is known as the Red Planet because of iron oxide on its surface.",
    "Jupiter is the largest planet in the Solar System.",
]
texts = [f"task: search result | query: {query}"] + [
    f"title: none | text: {document}" for document in documents
]
inputs = processor.tokenizer(texts, padding=True, return_tensors="np")
text_embeddings = model(
    **{key: mx.array(value) for key, value in inputs.items()}
).text_embeds
mx.eval(text_embeddings)

scores = np.asarray(text_embeddings[:1] @ text_embeddings[1:].T)[0]
for index in np.argsort(-scores):
    print(f"{scores[index]:.4f}  {documents[index]}")
print("Embedding shape:", text_embeddings.shape)

## 2. Matryoshka dimensions

Choose **128, 256, 512, or 768** dimensions. Keep the leading dimensions and
normalize again after truncation. Queries and documents must use the same dimension.
Smaller vectors reduce index storage; check retrieval quality on your own data.

In [ ]:
def matryoshka(embeddings, dimensions):
    if dimensions not in (128, 256, 512, 768):
        raise ValueError("Choose 128, 256, 512, or 768 dimensions.")
    vectors = embeddings[:, :dimensions]
    return vectors / mx.maximum(mx.linalg.norm(vectors, axis=-1, keepdims=True), 1e-12)


for dimensions in (128, 256, 512, 768):
    vectors = matryoshka(text_embeddings, dimensions)
    similarities = np.asarray(vectors[:1] @ vectors[1:].T)[0]
    best = int(np.argmax(similarities))
    print(
        f"{dimensions:3d} dimensions | cosine {similarities[best]:.4f} | {documents[best]}"
    )

## 3. Encode text in batches

The tokenizer pads each batch, and the model ignores padding when pooling.
Adjust `batch_size` to fit the available memory. The helpers below also evaluate
MLX's lazy arrays before returning results.

In [ ]:
def embed(inputs):
    vectors = model(
        **{key: mx.array(value) for key, value in inputs.items()}
    ).text_embeds
    mx.eval(vectors)
    return vectors


def embed_text(texts, batch_size=8):
    if not texts or batch_size < 1:
        raise ValueError("Provide at least one text and a positive batch size.")
    batches = []
    for start in range(0, len(texts), batch_size):
        inputs = processor.tokenizer(
            texts[start : start + batch_size], padding=True, return_tensors="np"
        )
        batches.append(embed(inputs))
    return mx.concatenate(batches, axis=0)


document_embeddings = embed_text(
    [f"title: none | text: {document}" for document in documents], batch_size=2
)
print("Batched document embeddings:", document_embeddings.shape)

## 4. Search images with text

Each inner list supplies the images belonging to one sample. Here, two images
produce two embeddings. `max_soft_tokens` controls the visual budget; supported
values are 70, 140, 280, 560, and 1120.

In [ ]:
image_paths = [
    EXAMPLES_DIR / "images" / name for name in ("cats.jpg", "desktop_setup.png")
]
images = [Image.open(path).convert("RGB") for path in image_paths]
image_inputs = processor(
    text=["title: none | text: <|image|>"] * len(images),
    images=[[image] for image in images],
    max_soft_tokens=280,
    return_tensors="mlx",
)
image_embeddings = embed(image_inputs)
image_query = embed_text(["task: search result | query: Two cats resting on a sofa."])
scores = np.asarray(image_query @ image_embeddings.T)[0]
for index in np.argsort(-scores):
    print(f"{scores[index]:.4f}  {image_paths[index].name}")
print("Image embeddings:", image_embeddings.shape)
preview = images[int(np.argmax(scores))].copy()
preview.thumbnail((480, 320))
preview

## 5. Embed an audio file

The pinned [public JFK speech clip](https://huggingface.co/datasets/Xenova/transformers.js-docs/blob/542437c29cc5273cfd29e25c77696f2b349badd7/jfk.wav)
contains spoken English. File audio is loaded and resampled by the processor's
existing audio loader. Replace `audio_path` with a local recording to try your own.

These scores compare the audio embedding with candidate descriptions; this does
not transcribe the recording.

In [ ]:
audio_path = hf_hub_download(
    repo_id="Xenova/transformers.js-docs",
    filename="jfk.wav",
    repo_type="dataset",
    revision="542437c29cc5273cfd29e25c77696f2b349badd7",
)
audio_inputs = processor(
    text="title: none | text: <|audio|>",
    audio=[audio_path],
    return_tensors="mlx",
)
audio_embedding = embed(audio_inputs)
audio_descriptions = [
    "A man giving a political speech.",
    "Instrumental piano music.",
    "A dog barking repeatedly.",
]
audio_queries = embed_text(
    [f"task: search result | query: {text}" for text in audio_descriptions]
)
scores = np.asarray(audio_embedding @ audio_queries.T)[0]
for index in np.argsort(-scores):
    print(f"{scores[index]:.4f}  {audio_descriptions[index]}")
print("Audio embedding:", audio_embedding.shape)

## 6. Video sampling and timestamps

Sample at one frame per second, then retain at most four frames spread across
the clip with `overflow_strategy="uniform"`. `"truncate"` keeps the earliest
sampled frames instead. `add_timestamps=True` includes frame times in the input.
Video embedding uses sampled frames; audio is a separate modality.

`return_metadata=True` exposes the selected indices and decoder backend.
Remove that diagnostic metadata before forwarding inputs to the model.
Compatible Apple Silicon/OpenCV builds use the Metal color-conversion path;
other builds and unsupported formats fall back to OpenCV.

In [ ]:
video_path = str(EXAMPLES_DIR / "videos" / "car_video.mp4")
video_inputs = processor(
    text="title: none | text: <|video|>",
    videos=[video_path],
    fps=1,
    max_frames=4,
    overflow_strategy="uniform",
    add_timestamps=True,
    max_soft_tokens=70,
    return_metadata=True,
    return_tensors="mlx",
)
video_metadata = video_inputs.pop("video_metadata")[0]
video_embedding = embed(video_inputs)
print("Video embedding:", video_embedding.shape)
print("Frame indices:", [int(index) for index in video_metadata.frames_indices])
print("Timestamps (seconds):", [round(float(t), 3) for t in video_metadata.timestamps])
print("Decoder:", video_metadata.video_backend)

## 7. Compose text and media into one embedding

Use the chat template to preserve content order and expand media placeholders.
This example combines a caption and image into a single vector. Audio and video
items can be included in the same content list using their `url` fields.

In [ ]:
conversation = [
    {"role": "system", "content": "title: none | text: "},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Two cats relaxing together indoors."},
            {"type": "image", "image": images[0]},
        ],
    },
]
composed_inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    return_dict=True,
    return_tensors="mlx",
    max_soft_tokens=280,
)
composed_embedding = embed(composed_inputs)
print("Caption + image:", composed_embedding.shape)

## 8. Batch different modalities together

Pass a list of conversations to obtain one vector per conversation, in the same
order. Here the batch contains text, an image, audio, and video. Longer inputs
increase padding and memory, so group similar lengths when scaling up.

In [ ]:
contents = [
    [{"type": "text", "text": "Two cats relaxing together indoors."}],
    [{"type": "image", "image": images[0]}],
    [{"type": "audio", "url": audio_path}],
    [{"type": "video", "url": video_path}],
]
conversations = [
    [
        {"role": "system", "content": "title: none | text: "},
        {"role": "user", "content": content},
    ]
    for content in contents
]
mixed_inputs = processor.apply_chat_template(
    conversations,
    tokenize=True,
    return_dict=True,
    return_tensors="mlx",
    fps=1,
    max_frames=4,
    max_soft_tokens=70,
)
mixed_embeddings = embed(mixed_inputs)
print("Rows: text, image, audio, video")
print("Mixed batch:", mixed_embeddings.shape)
print("Vector norms:", np.asarray(mx.linalg.norm(mixed_embeddings, axis=-1)))

## Optional: convert and reload an 8-bit model

Run this command in a terminal when you want a separate 8-bit copy. It writes
weights, the tokenizer, chat template, and processor settings to the output
folder. Conversion is optional and is not run automatically by this notebook.

```sh
mlx_vlm.convert --hf-path nativ-community/embeddinggemma-2 \
  --mlx-path embeddinggemma2-8bit --dtype bfloat16 --quantize --q-bits 8
```

Use your source model directory for `--hf-path` when working locally. Reload the
result with the same API, then rerun the examples:

```python
model, processor = load("./embeddinggemma2-8bit")
```

See the [model README](../mlx_vlm/models/embedding_gemma2/README.md) for requirements,
conversion notes, and decoder compatibility. Standard 4-bit quantization did not
pass the numerical accuracy checks.